In [6]:
"""
Cancer Prediction Pipeline — Inference Script (Updated)
Execution order: genomic features → Model 1 → inject pred_probs → Model 2 → Cox
"""

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

# ── Load saved objects ────────────────────────────────────────────────────────
model_dir = './saved_models'

scaler_m1          = joblib.load(f'{model_dir}/scaler_m1.pkl')
scaler_m2          = joblib.load(f'{model_dir}/scaler_m2.pkl')
pca                = joblib.load(f'{model_dir}/pca.pkl')
stage_encoder      = joblib.load(f'{model_dir}/stage_encoder.pkl')
gb_stage           = joblib.load(f'{model_dir}/gb_stage.pkl')
variant_models     = joblib.load(f'{model_dir}/variant_models.pkl')
variant_mean_probs = joblib.load(f'{model_dir}/variant_mean_probs.pkl')
top_variants       = joblib.load(f'{model_dir}/top_variants.pkl')
cph                = joblib.load(f'{model_dir}/cox_model.pkl')
risk_percentiles   = joblib.load(f'{model_dir}/risk_percentiles.pkl')
m1_feature_cols    = joblib.load(f'{model_dir}/m1_feature_cols.pkl')   # exact cols scaler_m1 was fit on
all_feature_cols   = joblib.load(f'{model_dir}/all_feature_cols.pkl')  # base clinical+genomic
all_feature_cols_m2 = joblib.load(f'{model_dir}/all_feature_cols_m2.pkl')  # base + pred_prob_*
race_encoder       = joblib.load(f'{model_dir}/race_encoder.pkl')

print(f"✅ Models loaded")
print(f"   Model 1 input cols : {len(m1_feature_cols)}")
print(f"   Model 1 variants   : {top_variants}")
print(f"   Model 2 total cols : {len(all_feature_cols_m2)}  "
      f"({len(all_feature_cols)} base + {len(top_variants)} Model 1 outputs)")
print(f"   Stage classes      : {list(stage_encoder.classes_)}")

# ── Default feature values ────────────────────────────────────────────────────
# GENOMIC — fed into Model 1 (scaler_m1 was fit on exactly these columns)
genomic_defaults = {
    'total_mutations'       : 150.0,
    'unique_genes'          : 80.0,
    'unique_chromosomes'    : 12.0,
    'tmb'                   : 150.0,
    'mean_VAF'              : 0.28,
    'max_VAF'               : 0.55,
    'std_VAF'               : 0.12,
    'mean_t_depth'          : 110.0,
    'mean_n_depth'          : 105.0,
    'mean_t_alt_count'      : 28.0,
    'mean_impact_ord'       : 1.8,
    'max_impact_ord'        : 3.0,
    'n_HIGH_impact'         : 5.0,
    'n_MODERATE_impact'     : 90.0,
    'n_LOW_impact'          : 30.0,
    'n_MODIFIER_impact'     : 25.0,
    'mean_pp_ord'           : 1.1,
    'mean_pp_score'         : 0.72,
    'n_probably_damaging'   : 35.0,
    'n_possibly_damaging'   : 20.0,
    'n_pp_benign'           : 30.0,
    'mean_sift_ord'         : 1.4,
    'mean_sift_score'       : 0.04,
    'n_deleterious_sift'    : 55.0,
    'n_tolerated_sift'      : 25.0,
    'has_stop_gained'       : 0,
    'has_frameshift'        : 0,
    'has_splice'            : 0,
    'has_utr_variant'       : 0,
    'has_intronic'          : 0,
    'has_downstream'        : 0,
    'n_synonymous_csq'      : 28.0,
    'n_nonsynonymous_csq'   : 90.0,
    'ratio_nonsyn_syn'      : 3.2,
    'n_snp'                 : 140.0,
    'n_indel'               : 10.0,
    'snp_fraction'          : 0.93,
    'n_cosmic_hits'         : 3.0,
    'n_rare_population'     : 135.0,
    'n_with_protein_domain' : 80.0,
    'domain_coverage_frac'  : 0.53,
    'mean_ncallers'         : 4.0,
    'max_ncallers'          : 5.0,
    # gene_* one-hot: all 0 → maps to OTHER bucket
}

# CLINICAL — fed into Model 2 alongside Model 1 output
clinical_defaults = {
    'Diagnosis Age'          : 60.0,
    'Fraction Genome Altered': 0.15,
    'Aneuploidy Score'       : 0.5,
    'Buffa Hypoxia Score'    : 0.0,
    'Ragnum Hypoxia Score'   : 0.0,
    'Winter Hypoxia Score'   : 0.0,
    'MSI MANTIS Score'       : 0.0,
    'MSIsensor Score'        : 0.0,
    'Mutation Count'         : 150.0,
    'TMB (nonsynonymous)'    : 4.5,
    'Sex_encoded'            : 0,    # 0=Female, 1=Male
    'Race_encoded'           : 0,
}


# ── Inference function ────────────────────────────────────────────────────────
def predict(genomic_features: dict, clinical_features: dict, verbose: bool = True) -> dict:
    """
    Stage 1: m1_feature_cols (genomic only) → scaler_m1 → variant_models → pred_prob_*
    Stage 2: all_feature_cols_m2 (base + pred_prob_*) → scaler_m2 → pca → gb_stage + cph
    """

    # ── Stage 1: Model 1 ─────────────────────────────────────────────────────
    # Build row using ONLY the columns scaler_m1 was fit on
    m1_row = {col: genomic_features.get(col, 0.0) for col in m1_feature_cols}
    X_m1 = pd.DataFrame([m1_row])[m1_feature_cols].fillna(0.0)

    X_m1_scaled = scaler_m1.transform(X_m1)   # no column mismatch — exact fit columns used

    variant_probs = []
    for i, model in enumerate(variant_models):
        if model is None:
            prob = float(variant_mean_probs[i])
        else:
            prob = float(model.predict_proba(X_m1_scaled)[0, 1])
        variant_probs.append(prob)

    pred_prob_dict = {f'pred_prob_{top_variants[i]}': variant_probs[i]
                      for i in range(len(top_variants))}

    if verbose:
        print("\n[Model 1] Predicted Variant Probabilities (from genomic features):")
        for k, v in pred_prob_dict.items():
            print(f"  {k.replace('pred_prob_', ''):35s} {v:.4f}")

    # ── Stage 2: Model 2 ─────────────────────────────────────────────────────
    # Merge: base features (genomic + clinical) + Model 1 outputs
    combined = {**genomic_features, **clinical_features, **pred_prob_dict}
    m2_row = {col: combined.get(col, 0.0) for col in all_feature_cols_m2}
    X_m2 = pd.DataFrame([m2_row])[all_feature_cols_m2].fillna(0.0)

    X_m2_scaled = scaler_m2.transform(X_m2)
    X_pca = pca.transform(X_m2_scaled)

    # Stage prediction
    stage_pred_enc   = gb_stage.predict(X_pca[:, :25])[0]
    stage_pred       = stage_encoder.inverse_transform([stage_pred_enc])[0]
    stage_proba      = gb_stage.predict_proba(X_pca[:, :25])[0]
    stage_confidence = float(np.max(stage_proba))
    stage_proba_dict = {stage_encoder.classes_[i]: float(stage_proba[i])
                        for i in range(len(stage_encoder.classes_))}

    # Cox survival risk
    cox_input  = pd.DataFrame(X_pca[:, :15], columns=[f'PC_{i}' for i in range(15)])
    risk_score = float(cph.predict_partial_hazard(cox_input).values[0])
    risk_group = ('Low'    if risk_score <= risk_percentiles[0] else
                  'Medium' if risk_score <= risk_percentiles[1] else 'High')

    return {
        'model1_variant_probabilities': pred_prob_dict,
        'predicted_stage'             : stage_pred,
        'stage_confidence'            : stage_confidence,
        'stage_probabilities'         : stage_proba_dict,
        'cox_risk_score'              : risk_score,
        'risk_group'                  : risk_group,
    }


# ── Run with defaults ─────────────────────────────────────────────────────────
results = predict(genomic_defaults, clinical_defaults, verbose=True)

print("\n" + "="*60)
print("CANCER PREDICTION RESULTS")
print("="*60)

print("\n[Model 2] Cancer Stage:")
print(f"  Predicted : {results['predicted_stage']}")
print(f"  Confidence: {results['stage_confidence']:.2%}")
print("  Per-class probabilities:")
for stage, prob in results['stage_probabilities'].items():
    bar = '█' * int(prob * 30)
    print(f"    {stage:12s} {prob:.4f}  {bar}")

print(f"\n[Model 3] Survival (Cox):")
print(f"  Risk score: {results['cox_risk_score']:.4f}")
print(f"  Risk group: {results['risk_group']}")

print("\n" + "="*60)
print("NOTE: gene_* one-hot cols default to 0 (unknown gene → OTHER).")
print("pred_prob_* cols are computed by Model 1 — do not set manually.")
print("="*60)

✅ Models loaded
   Model 1 input cols : 64
   Model 1 variants   : ['Missense_Mutation', 'Silent', "3'UTR", "5'UTR", 'Nonsense_Mutation', 'Intron', 'Frame_Shift_Del', 'Splice_Site']
   Model 2 total cols : 84  (76 base + 8 Model 1 outputs)
   Stage classes      : ['STAGE I', 'STAGE II', 'STAGE III', 'STAGE IIIA', 'STAGE IIIB', 'STAGE IIIC', 'STAGE IV', 'STAGE IVA', 'STAGE IVB']

[Model 1] Predicted Variant Probabilities (from genomic features):
  Missense_Mutation                   1.0000
  Silent                              1.0000
  3'UTR                               0.5130
  5'UTR                               0.2050
  Nonsense_Mutation                   0.0000
  Intron                              0.0000
  Frame_Shift_Del                     0.0450
  Splice_Site                         0.0000

CANCER PREDICTION RESULTS

[Model 2] Cancer Stage:
  Predicted : STAGE IIIB
  Confidence: 59.07%
  Per-class probabilities:
    STAGE I      0.1843  █████
    STAGE II     0.1361  ████
    S